In [1]:
%cd ..

c:\Users\HP\OneDrive - University of Moratuwa\Desktop\E-Vision-Projects\DB_SQL_GEN


In [2]:
from src.config import settings
from pathlib import Path
import yaml
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_chroma import Chroma
import os
from typing import List, Dict

c:\Users\HP\OneDrive - University of Moratuwa\Desktop\E-Vision-Projects\DB_SQL_GEN\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
_YMAL_PATH = Path(settings.business_doc_ymal_path)

data = yaml.safe_load(_YMAL_PATH.read_text(encoding="utf-8"))

In [4]:
data.get('definitions').get('net_sales')

{'keywords': ['net sales', 'net sale', 'achievement', 'revenue', 'sale'],
 'definition': "Net Sales:\n  Sales ('sales') minus return ('return') taken as net sale.\n  also calling as achievement\n  Covers both sales and return in one calculation.\n"}

In [5]:
class BusinessKnowledgeStore:
    """
    Manages a vector store of business definitions.
    """
    def __init__(self, yaml_path: str = None):
        
        self.yaml_path = yaml_path or settings.business_doc_ymal_path
        self.business_info_dict = self._load_yaml()
        
        # Initialize embeddings with HuggingFace model (local)
        self.embeddings = HuggingFaceEmbeddings(
            model_name=settings.embedding_model
        )
        
        # Initialize vector store
        persist_directory = settings.vector_store_path
        os.makedirs(persist_directory, exist_ok=True)
        
        self.vectorstore = Chroma(
            collection_name="business_definition",
            embedding_function=self.embeddings,
            persist_directory=persist_directory
        )
        
        
    def _load_yaml(self):
        data = yaml.safe_load(
            Path(self.yaml_path).read_text(encoding='utf-8')
            )
        return data.get("definitions", {})
    
    def seed_business_definitions(self, persist_dir, collection_name):
        
        docs = []
        for name, entry in self.business_info_dict.items():
            docs.append(
                Document(
                    page_content=entry["definition"].strip(),
                    metadata={
                        "name": name,
                        "keywords": entry.get("keywords", [])
                    }
                )
            )
            
        self.vectorstore.add_documents(docs)
        
    def get_relevant_definition_docs(self, question: str, k: int = 3) -> List[Dict]:
        
        docs = self.vectorstore.similarity_search(question, k=k)
        
        # results = self.vectorstore.similarity_search_with_score(question, k=k)
        
        # threshold = 0.7   # tune this
        # docs = []
        
        # for doc , score in results:
        #     print(score)
        #     if score > threshold:
        #         docs.append(doc)
            
        return docs
    
    def retrieve_business_definitions_block(self, question: str, k: int = 3) -> str:
        
        docs = self.get_relevant_definition_docs(question, k)
        
        lines = []
        for doc in docs:
            # name = doc.metadata.get("name", "---").replace("_", " ")
            # lines.append(f"{name}:")
            lines.append(doc.page_content.strip())
            lines.append("")
            
        return "\n".join(lines).strip()
        

In [6]:
bks = BusinessKnowledgeStore()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14715.35it/s]


In [7]:
bks.seed_business_definitions("", "")

In [8]:
docs = bks.get_relevant_definition_docs("what is my achievement")

In [9]:
docs

[Document(id='35650394-12d3-4e1c-bd13-605e699d3eb7', metadata={'keywords': ['net sales', 'net sale', 'achievement', 'revenue', 'sale'], 'name': 'net_sales'}, page_content="Net Sales:\n  Sales ('sales') minus return ('return') taken as net sale.\n  also calling as achievement\n  Covers both sales and return in one calculation."),
 Document(id='715e7e86-63d1-4f81-9385-cd71c8b606c0', metadata={'keywords': ['vs target', 'achievement vs', 'target vs', 'against target', 'target achievement'], 'name': 'achievement_vs_target'}, page_content='Achievement vs Target - CRITICAL two-step rule:\n  Step 1: Calculate Net Sales for the rep independently.\n  Step 2: Calculate Target for the rep independently.\n  Step 3: Join both results at rep level AFTER aggregation.\n  NEVER aggregate sales and targets in the same step - causes row multiplication.'),
 Document(id='fdbaa35c-dd9f-40f9-92f8-4c1be96b37dd', metadata={'keywords': ['run rate', 'required rate', 'daily rate', 'with run'], 'name': 'run_rate'},

In [ ]:
# def format_definitions_for_prompt(docs):
#     lines = []
#     for doc in docs:
#         # name = doc.metadata.get("name", "---").replace("_", " ")
#         # lines.append(f"{name}:")
#         lines.append(doc.page_content.strip())
#         lines.append("")
        
#     return "\n".join(lines).strip()

In [ ]:
# out = format_definitions_for_prompt(docs)
# print(out)

Achievement vs Target - CRITICAL two-step rule:
  Step 1: Calculate Net Sales for the rep independently.
  Step 2: Calculate Target for the rep independently.
  Step 3: Join both results at rep level AFTER aggregation.
  NEVER aggregate sales and targets in the same step - causes row multiplication.

Run Rate:
  Current run rate  = Achievement divided by days elapsed in period.
  Required run rate = (Target minus Achievement) divided by remaining days.

Net Sales (also called Achievement):
  Formula: SUM(CASE WHEN Type='sales' THEN in_sales ELSE -in_sales END)
  Covers both sales and returns in one calculation.


In [10]:
block = bks.retrieve_business_definitions_block("what is my achievement")

In [11]:
print(block)

Net Sales:
  Sales ('sales') minus return ('return') taken as net sale.
  also calling as achievement
  Covers both sales and return in one calculation.

Achievement vs Target - CRITICAL two-step rule:
  Step 1: Calculate Net Sales for the rep independently.
  Step 2: Calculate Target for the rep independently.
  Step 3: Join both results at rep level AFTER aggregation.
  NEVER aggregate sales and targets in the same step - causes row multiplication.

Run Rate:
  Current run rate  = Achievement divided by days elapsed in period.
  Required run rate = (Target minus Achievement) divided by remaining days.
